# JSON Output Structure usoing langchain

In [8]:
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
from dotenv import load_dotenv
load_dotenv()


True

In [ ]:
# opld model 
class AnswerWithJustification(BaseModel):
    """An answer to the users questions along with justification for the answers"""
    answer :str
    """Justification for the answer"""
    justification : str
    

In [4]:
llm = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature=0.7
)

structure_llm = llm.with_structured_output(AnswerWithJustification)

result = structure_llm.invoke("""What weighs more, a pound of bricks or a pound 
    of feathers""")

In [6]:
print(result)

answer='They weigh the same' justification='A pound is a unit of weight or mass, so a pound of bricks and a pound of feathers both weigh the same amount, which is one pound. The difference is in their density and the amount of space they occupy, with bricks being much denser than feathers.'


In [9]:
# NEW model with baseModel and Field
class UserInfo(BaseModel):
    name: str = Field(description='Name of the person')
    age: int = Field(description='Age of the person')
    skill: list[str] = Field(description='List of Programming skill')
    
structured_llm = llm.with_structured_output(UserInfo)


In [11]:
# Now, the response is a Python Object, not a string!
response = structured_llm.invoke("My name is Aryan, I am 22 and I know Python and Java")

print(response.name)   # Output: Aryan
print(response.skill) # Output: ['Python', 'Java']

Aryan
['Python', 'Java']


## Output Parsers (The Traditional Way)

In [13]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = "List 3 fruit and theire color in Json format"

chain = llm | parser

result = chain.invoke(prompt)

print(result)

{'fruits': [{'name': 'Apple', 'color': 'Red'}, {'name': 'Banana', 'color': 'Yellow'}, {'name': 'Grapes', 'color': 'Purple'}]}


## 3. CommaSeparatedListOutputParser

In [14]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

item = parser.invoke("apple, banana, cherry")
print(item)

['apple', 'banana', 'cherry']


In [ ]:
# here use promt to listoutput 
prompt = 'List of 3 color and give with specific fruit'
chain = llm | parser
result = chain.invoke(prompt)

print(result)

['Here are 3 colors with a specific fruit associated with each:', '1. **Red** - Strawberry', '2. **Yellow** - Banana', '3. **Orange** - Orange']


## PydanticOutputParser (The Manual Way)


In [16]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

In [19]:
# 1. Define the Blueprint (Schema)
class Person(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person")

In [22]:
# 1. Setup Parser
parser = PydanticOutputParser(pydantic_object=Person)

# 2. Add instructions to Prompt
prompt = PromptTemplate(
    template="Extract info.\n{format_instructions}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 3. Create Chain (Prompt -> LLM -> Parser)
chain = prompt | llm | parser
result = chain.invoke({"query": "Anjali is 24."})
print(result) 

name='Anjali' age=24


# Imperative Composition

Imperative composition is just a fancy name for writing the code you’re used to
writing, composing these components into functions and classes. Here’s an example
combining prompts, models, and output parsers:

In [ ]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

In [24]:
template = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
('human', '{question}'),
])

@chain
def chatbot(values):
    prompt = template.invoke(values)
    return llm.invoke(prompt)

chatbot.invoke({'question':"Which model provider offers LLMs"})

AIMessage(content='Several model providers offer Large Language Models (LLMs). Here are some of the notable ones:\n\n1. **Hugging Face**: Hugging Face provides a wide range of pre-trained LLMs, including BERT, RoBERTa, and XLNet, through their Transformers library.\n2. **Google**: Google offers LLMs like BERT, ALBERT, and T5 through their TensorFlow and TensorFlow Hub platforms.\n3. **Microsoft**: Microsoft provides LLMs like Turing-NLG and DeBERTa through their Azure Machine Learning and GitHub repositories.\n4. **Meta AI**: Meta AI (formerly Facebook AI) offers LLMs like LLaMA and OPT through their GitHub repository.\n5. **Stanford Natural Language Processing Group**: The Stanford NLP Group provides pre-trained LLMs like ELMo and CoVe through their GitHub repository.\n6. **OpenAI**: OpenAI offers LLMs like GPT-3 and GPT-2 through their API platform.\n7. **Amazon**: Amazon provides LLMs like BERT and RoBERTa through their SageMaker platform.\n8. **IBM**: IBM offers LLMs like Watson Na

# Declarative Composition
LCEL is a declarative language for composing LangChain components. LangChain
compiles LCEL compositions to an optimized execution plan, with automatic paralleli
zation, streaming, tracing, and async support.
Let’s see the same example using LCEL:

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
# the building blocks
template = ChatPromptTemplate.from_messages([
('system', 'You are a helpful assistant.'),
('human', '{question}'),
])

In [27]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature=0.7
)


In [30]:
# combine them with the | operator
chatbot = template | llm
# use it
chatbot.invoke({"question": "Which model providers offer LLMs?"})

AIMessage(content="The landscape of Large Language Model (LLM) providers is rapidly evolving, with new players emerging and existing ones expanding their offerings. Here are the major model providers that offer LLMs, often through APIs, cloud platforms, or as downloadable open-source models:\n\n1.  **OpenAI:**\n    *   **Models:** GPT-3.5, GPT-4 (and various specialized versions like GPT-4 Turbo), DALL-E (for image generation, often integrated).\n    *   **Access:** Primarily through their API, and via consumer products like ChatGPT.\n    *   **Note:** The pioneer in bringing powerful LLMs to the mainstream, widely used for a vast array of applications.\n\n2.  **Google (Google AI / Google Cloud):**\n    *   **Models:** Gemini (Ultra, Pro, Nano), PaLM 2, LaMDA (older, conversational focus), Imagen (for image generation).\n    *   **Access:** Google Cloud's Vertex AI platform, API access, and consumer products like the Gemini web interface.\n    *   **Note:** Deep research capabilities, 